# AI TRAINING – WEEK 3
## DAY 4 ASSIGNMENT: Data Cleaning & Preprocessing
### Missing Values, Duplicates, Types, Outliers & Encoding

- **Student Name:** Lasya Kolluru
- **GitHub Username:** Lasya-Kolluru
- **Email:** kollurlasyachowdary@gmail.com
- **Deliverable:** `Week3_Day4_Data_Cleaning_Lasya_Kolluru.ipynb` & `cleaned_superstore_sales.csv`
- **Dataset Name:** Superstore Retail Sales Dataset (`superstore_sales.csv`)
- **Dataset Source:** Tableau Public / Kaggle Public Datasets
- **Track:** Data
- **Topic Area:** Data cleaning: missing values, duplicates, types, outliers, encoding
- **Learning Goals:** Understand data cleaning: missing values, duplicates, types, outliers, encoding; explain the key concepts in your own words; apply them in the practical exercise; identify common errors and improve the solution.
- **Hands-On Activity:** Clean a messy customer dataset
- **Daily Deliverable:** Clean dataset + before/after report


---
## Core Definitions & Concepts (Simple & Easy to Remember)

1. **Data Cleaning:** The process of detecting, correcting, or removing corrupt, incomplete, duplicate, or improperly formatted records from a dataset.
2. **Missing Values (`NaN`):** Unrecorded or blank data points. Handled either by removal (`dropna`) or imputation with statistical values like median or mode (`fillna`).
3. **Duplicates:** Repeated records that artificially inflate observations and distort metrics. Handled using `drop_duplicates()`.
4. **Data Type Casting:** Ensuring columns have appropriate operational formats (e.g., converting text numbers to `float` and string dates to `datetime`).
5. **Outliers:** Extreme values unusually far from the rest of the data. Often detected using the **Interquartile Range (IQR)** rule and capped or removed.
6. **Encoding:** Converting non-numeric text categories into numbers so machine learning models can process them (e.g., One-Hot Encoding and Label Encoding).


---
## Part A – Ingesting & Simulating a Messy Customer Dataset

In real-world data science, raw datasets are rarely clean. Here we load `superstore_sales.csv` and inject realistic real-world data issues (missing entries, duplicate orders, irregular casing, dirty text numbers, and extreme outliers) to build a robust cleaning pipeline.


In [1]:
import pandas as pd
import numpy as np

# Load raw base dataset
raw_df = pd.read_csv("superstore_sales.csv")
np.random.seed(42)

# Create a realistic messy version to demonstrate end-to-end cleaning
messy_df = raw_df.copy()

# 1. Inject duplicate rows (25 duplicate transactions)
duplicates = messy_df.sample(25, random_state=42)
messy_df = pd.concat([messy_df, duplicates], ignore_index=True)

# 2. Inject missing values (NaNs) in Sales, Discount, and Segment
missing_indices_sales = np.random.choice(messy_df.index, size=35, replace=False)
missing_indices_discount = np.random.choice(messy_df.index, size=20, replace=False)
missing_indices_segment = np.random.choice(messy_df.index, size=15, replace=False)

messy_df.loc[missing_indices_sales, "Sales"] = np.nan
messy_df.loc[missing_indices_discount, "Discount"] = np.nan
messy_df.loc[missing_indices_segment, "Segment"] = np.nan

# 3. Inject inconsistent text whitespace and casing in Category
messy_df["Category"] = messy_df["Category"].apply(
    lambda x: f"  {x.lower()} " if np.random.rand() > 0.7 else x
)

# 4. Inject extreme anomalous outliers in Sales
outlier_indices = np.random.choice(messy_df.index, size=5, replace=False)
messy_df.loc[outlier_indices, "Sales"] = [85000.0, 92000.0, 78000.0, -1500.0, 99999.0]

print(f"Messy Dataset Created: {messy_df.shape[0]} rows, {messy_df.shape[1]} columns.")
print(f"Missing Values: Sales={messy_df['Sales'].isna().sum()}, Discount={messy_df['Discount'].isna().sum()}, Segment={messy_df['Segment'].isna().sum()}")
print(f"Duplicate Rows Count: {messy_df.duplicated().sum()}")


Messy Dataset Created: 1025 rows, 20 columns.
Missing Values: Sales=35, Discount=20, Segment=15
Duplicate Rows Count: 11


---
## Part B – Practical Cleaning Exercise

### Step 1: Detecting and Removing Duplicates
Duplicate rows distort business counts, revenue metrics, and averages.


In [2]:
initial_rows = len(messy_df)
dup_count = messy_df.duplicated().sum()
print(f"Identified duplicate rows: {dup_count}")

# Remove duplicates keeping the first occurrence
cleaned_df = messy_df.drop_duplicates(keep='first').copy()
print(f"Rows before: {initial_rows} -> Rows after removing duplicates: {len(cleaned_df)}")


Identified duplicate rows: 11
Rows before: 1025 -> Rows after removing duplicates: 1014


### Step 2: Inspecting and Imputing Missing Values
- For **numerical columns with outliers or skewness** (`Sales`), **median** imputation is preferred over mean because the median is robust to extreme values.
- For **numerical rates** (`Discount`), the **median** is also suitable.
- For **categorical columns** (`Segment`), we impute using the **mode** (most frequent class).


In [3]:
print("Missing values before imputation:")
print(cleaned_df[['Sales', 'Discount', 'Segment']].isna().sum())

# Impute numerical Sales using median
sales_median = cleaned_df['Sales'].median()
cleaned_df['Sales'] = cleaned_df['Sales'].fillna(sales_median)

# Impute numerical Discount using median
discount_median = cleaned_df['Discount'].median()
cleaned_df['Discount'] = cleaned_df['Discount'].fillna(discount_median)

# Impute categorical Segment using mode
segment_mode = cleaned_df['Segment'].mode()[0]
cleaned_df['Segment'] = cleaned_df['Segment'].fillna(segment_mode)

print("")
print("Missing values after imputation:")
print(cleaned_df[['Sales', 'Discount', 'Segment']].isna().sum())
print(f"Imputed Sales using median: ${sales_median:.2f}")
print(f"Imputed Discount using median: {discount_median}")
print(f"Imputed Segment using mode: '{segment_mode}'")


Missing values before imputation:
Sales       35
Discount    20
Segment     15
dtype: int64

Missing values after imputation:
Sales       0
Discount    0
Segment     0
dtype: int64
Imputed Sales using median: $397.33
Imputed Discount using median: 0.1
Imputed Segment using mode: 'Consumer'


### Step 3: Text Standardization and Type Casting
Inconsistent string casing (`"furniture "` vs `"Furniture"`) causes grouping errors. We strip whitespace, standardize casing, and ensure dates are in proper datetime format.


In [4]:
# Inspect category uniqueness before standardizing
print("Categories before cleaning:", repr(cleaned_df['Category'].unique().tolist()))

# Clean string columns: strip whitespace and apply Title Case
cleaned_df['Category'] = cleaned_df['Category'].astype(str).str.strip().str.title()
cleaned_df['Segment'] = cleaned_df['Segment'].astype(str).str.strip().str.title()
cleaned_df['Ship Mode'] = cleaned_df['Ship Mode'].astype(str).str.strip().str.title()

# Convert Order Date and Ship Date to datetime
cleaned_df['Order Date'] = pd.to_datetime(cleaned_df['Order Date'])
cleaned_df['Ship Date'] = pd.to_datetime(cleaned_df['Ship Date'])

# Convert Postal Code to string (ZIP codes are identifiers, not numerical values)
cleaned_df['Postal Code'] = cleaned_df['Postal Code'].astype(str)

print("Categories after cleaning:", cleaned_df['Category'].unique().tolist())
print(f"Order Date dtype: {cleaned_df['Order Date'].dtype}")
print(f"Postal Code dtype: {cleaned_df['Postal Code'].dtype}")


Categories before cleaning: ['Furniture', 'Office Supplies', 'Technology', '  furniture ', '  office supplies ', '  technology ']
Categories after cleaning: ['Furniture', 'Office Supplies', 'Technology']
Order Date dtype: datetime64[us]
Postal Code dtype: str


### Step 4: Outlier Detection and Treatment (IQR Method)
Using the **Interquartile Range (IQR)**:
- $Q_1$ = 25th percentile, $Q_3$ = 75th percentile, $IQR = Q_3 - Q_1$
- Lower Bound = $Q_1 - 1.5 \times IQR$ (clamped to 0 for non-negative sales)
- Upper Bound = $Q_3 + 1.5 \times IQR$
Outliers beyond these thresholds are capped (Winsorized) to preserve observations while preventing model distortion.


In [5]:
Q1 = cleaned_df['Sales'].quantile(0.25)
Q3 = cleaned_df['Sales'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = max(0.0, Q1 - 1.5 * IQR)
upper_bound = Q3 + 1.5 * IQR

outliers_count = ((cleaned_df['Sales'] < lower_bound) | (cleaned_df['Sales'] > upper_bound)).sum()
print(f"Q1: ${Q1:.2f}, Q3: ${Q3:.2f}, IQR: ${IQR:.2f}")
print(f"Acceptable range: [${lower_bound:.2f}, ${upper_bound:.2f}]")
print(f"Identified outliers in Sales: {outliers_count}")

# Cap (Winsorize) outliers to bounds
cleaned_df['Sales_Capped'] = cleaned_df['Sales'].clip(lower=lower_bound, upper=upper_bound)
print(f"Max Sales before capping: ${cleaned_df['Sales'].max():,.2f}")
print(f"Max Sales after capping: ${cleaned_df['Sales_Capped'].max():,.2f}")
cleaned_df['Sales'] = cleaned_df['Sales_Capped']
cleaned_df.drop(columns=['Sales_Capped'], inplace=True)


Q1: $179.48, Q3: $818.88, IQR: $639.40
Acceptable range: [$0.00, $1777.97]
Identified outliers in Sales: 91
Max Sales before capping: $99,999.00
Max Sales after capping: $1,777.97


### Step 5: Encoding Categorical Variables
Machine learning models require numerical matrices:
1. **One-Hot Encoding:** Creates binary columns (0 or 1) for nominal categories without inherent order (e.g., `Segment`: Consumer, Corporate, Home Office).
2. **Label / Ordinal Encoding:** Maps categories to sequential integers (0, 1, 2...).


In [6]:
# 1. One-Hot Encoding on Segment
segment_ohe = pd.get_dummies(cleaned_df['Segment'], prefix='Segment', dtype=int)
print("One-Hot Encoded Segment (First 3 rows):")
print(segment_ohe.head(3))

# 2. Label Encoding on Category
category_mapping = {cat: i for i, cat in enumerate(cleaned_df['Category'].unique())}
cleaned_df['Category_Encoded'] = cleaned_df['Category'].map(category_mapping)

print("")
print("Category Label Encoding Mapping:", category_mapping)
print(cleaned_df[['Category', 'Category_Encoded']].head(3))


One-Hot Encoded Segment (First 3 rows):
   Segment_Consumer  Segment_Corporate  Segment_Home Office
0                 0                  1                    0
1                 0                  1                    0
2                 1                  0                    0

Category Label Encoding Mapping: {'Furniture': 0, 'Office Supplies': 1, 'Technology': 2}
          Category  Category_Encoded
0        Furniture                 0
1  Office Supplies                 1
2  Office Supplies                 1


---
## Part C – Before / After Data Cleaning Quality Report

Here we compare the state of the raw messy dataset against the final cleaned dataset.


In [7]:
report = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Duplicate Rows',
        'Missing Values (Total)',
        'Sales Min Value',
        'Sales Max Value',
        'Sales Mean Value',
        'Inconsistent Categories Count',
        'Order Date Data Type'
    ],
    'Before Cleaning': [
        len(messy_df),
        messy_df.duplicated().sum(),
        messy_df.isna().sum().sum(),
        f"${messy_df['Sales'].min():,.2f}",
        f"${messy_df['Sales'].max():,.2f}",
        f"${messy_df['Sales'].mean():,.2f}",
        len(messy_df['Category'].unique()),
        str(messy_df['Order Date'].dtype)
    ],
    'After Cleaning': [
        len(cleaned_df),
        cleaned_df.duplicated().sum(),
        cleaned_df.isna().sum().sum(),
        f"${cleaned_df['Sales'].min():,.2f}",
        f"${cleaned_df['Sales'].max():,.2f}",
        f"${cleaned_df['Sales'].mean():,.2f}",
        len(cleaned_df['Category'].unique()),
        str(cleaned_df['Order Date'].dtype)
    ]
})

print("==================== BEFORE / AFTER DATA CLEANING REPORT ====================")
print(report.to_string(index=False))


==================== BEFORE / AFTER DATA CLEANING REPORT ====================
                       Metric Before Cleaning After Cleaning
                   Total Rows            1025           1014
               Duplicate Rows              11              9
       Missing Values (Total)              70              0
              Sales Min Value      $-1,500.00          $0.00
              Sales Max Value      $99,999.00      $1,777.97
             Sales Mean Value       $1,016.43        $579.11
Inconsistent Categories Count               6              3
         Order Date Data Type             str datetime64[us]


### Step 7: Exporting the Cleaned Dataset
We export the final sanitized dataset to `cleaned_superstore_sales.csv` for downstream modeling and visualization.


In [8]:
output_csv = "cleaned_superstore_sales.csv"
cleaned_df.to_csv(output_csv, index=False)
print(f"Cleaned dataset exported successfully: {output_csv}")
print(f"Final Clean Shape: {cleaned_df.shape[0]} rows, {cleaned_df.shape[1]} columns.")


Cleaned dataset exported successfully: cleaned_superstore_sales.csv
Final Clean Shape: 1014 rows, 21 columns.


---
## Part D – Common Errors in Data Cleaning & Troubleshooting

### 1. `SettingWithCopyWarning`
- **Cause:** Attempting to modify a slice of a DataFrame directly instead of creating an explicit copy using `.copy()`.
- **Solution:** Always use `df.loc[row_indexer, col_indexer] = value` or append `.copy()` when slicing.

### 2. Imputing Skewed Data with Mean instead of Median
- **Cause:** Calculating `mean()` when extreme outliers exist drags the imputed value away from typical business transactions.
- **Solution:** Use `median()` for skewed continuous variables and `mode()` for categorical strings.

### 3. Dummy Variable Trap in One-Hot Encoding
- **Cause:** Including all $K$ dummy columns when a constant bias term is present creates perfect multicollinearity.
- **Solution:** Pass `drop_first=True` in `pd.get_dummies()` for linear regression models.

### 4. Overlooking Hidden Whitespace
- **Cause:** Values like `'Furniture'` and `'Furniture '` are treated as separate categories by `groupby()`.
- **Solution:** Always call `.str.strip()` on text columns before grouping or filtering.


In [9]:
print("--- Demonstrating Whitespace Grouping Error & Solution ---")
sample_data = pd.DataFrame({'Category': ['Furniture', 'Furniture ', 'Technology'], 'Sales': [100, 200, 300]})
print("Grouping before .str.strip() (Faulty 3 categories):")
print(sample_data.groupby('Category')['Sales'].sum())

sample_data['Category'] = sample_data['Category'].str.strip()
print("")
print("Grouping after .str.strip() (Correct 2 categories):")
print(sample_data.groupby('Category')['Sales'].sum())


--- Demonstrating Whitespace Grouping Error & Solution ---
Grouping before .str.strip() (Faulty 3 categories):
Category
Furniture     100
Furniture     200
Technology    300
Name: Sales, dtype: int64

Grouping after .str.strip() (Correct 2 categories):
Category
Furniture     300
Technology    300
Name: Sales, dtype: int64


---
## Part E – Key Concepts Explained

### 1. What is the difference between dropping missing values and imputing them?
**Answer:** Dropping (`dropna`) deletes rows completely, which risks losing valuable data. Imputing (`fillna`) fills in blanks using smart estimates (like median or mode) to keep the full dataset intact.

### 2. Why is the Median preferred over the Mean for imputing skewed numbers?
**Answer:** The mean gets pulled by extreme outliers (like a $90,000 order), while the median stays stable right in the true center of normal orders.

### 3. How do you find outliers using the IQR rule?
**Answer:** Find the middle 50% range ($IQR = Q_3 - Q_1$). Any value lower than $Q_1 - 1.5 \times IQR$ or higher than $Q_3 + 1.5 \times IQR$ is flagged as an outlier.

### 4. What is the difference between One-Hot Encoding and Label Encoding?
**Answer:** 
- **One-Hot Encoding:** Creates separate 0/1 columns for each category (best for unordered names like colors or segments).
- **Label Encoding:** Assigns sequential numbers like 0, 1, 2 to categories (best for ranked data like Small, Medium, Large).

### 5. Why should Postal Codes be stored as strings instead of integers?
**Answer:** Postal codes are identity labels, not measurable math quantities. Storing them as integers drops leading zeros (e.g., `01234` becomes `1234`).


---
## Student Submission Checklist
- [x] Loaded raw `superstore_sales.csv` dataset.
- [x] Identified and removed duplicate transaction records.
- [x] Handled missing values using median (numerical) and mode (categorical) imputation.
- [x] Cleaned text formatting with `.str.strip()` and converted dates with `pd.to_datetime()`.
- [x] Detected and treated outliers using the 1.5 × IQR method.
- [x] Implemented One-Hot Encoding and Label Encoding on categorical features.
- [x] Generated a clear Before/After data quality comparison report.
- [x] Exported the clean dataset to `cleaned_superstore_sales.csv`.
- [x] Explained key concepts and common errors in simple, memorable words.
